# Predicción de la Progresión de la Diabetes — Fase Exploratoria / MLOps

### Objetivo
Construir un modelo de regresión para predecir la progresión de la enfermedad de diabetes con un año de diferencia respecto a la línea base, a partir de características numéricas de pacientes.

### Tipo de problema
Este es un problema de **regresión**, ya que se predice una medida cuantitativa continua (progresión de la enfermedad).

### Flujo del notebook
Este cuaderno se ejecuta de extremo a extremo en **una única celda de código** (formato listo para **Google Colab**) e incluye:
1. Carga del dataset.
2. Limpieza de datos (misma política que el pipeline MLOps).
3. División en entrenamiento / validación / test.
4. Entrenamiento de candidatos y validación cruzada.
5. Evaluación final, selección del mejor candidato y visualización básica.

> Nota: el flujo productivo (scripts, MLflow, API, CI/CD) vive en el repositorio MLOps; este cuaderno es la fase exploratoria inicial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")

# ---------- 1. Carga ----------
diabetes = load_diabetes()
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df["target"] = diabetes.target

print(f"Shape del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Columnas: {df.columns.tolist()}")
print("\nEstadisticas descriptivas:")
print(df.describe().round(2))

# ---------- 2. Limpieza (misma politica que el pipeline MLOps) ----------
df_clean = df.copy()
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
df_clean = df_clean.dropna(subset=["target"])
df_clean = df_clean.loc[df_clean["target"] > 0]
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# ---------- 3. Division ----------
X = df_clean.drop(columns=["target"])
y = df_clean["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
X_train_fit, X_validation, y_train_fit, y_validation = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, shuffle=True
)

# ---------- 4. Candidatos + validacion cruzada + evaluacion final ----------
def make_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

candidates = {
    "linear_regression": make_pipeline(LinearRegression()),
    "random_forest": make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    "gradient_boosting": make_pipeline(GradientBoostingRegressor(random_state=42)),
}

results = []
for name, model in candidates.items():
    model.fit(X_train_fit, y_train_fit)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_predictions = cross_val_predict(model, X_train_fit, y_train_fit, cv=cv)
    test_predictions = model.predict(X_test)
    results.append({
        "modelo": name,
        "rmse_cv": float(np.sqrt(mean_squared_error(y_train_fit, cv_predictions))),
        "mae_cv": float(mean_absolute_error(y_train_fit, cv_predictions)),
        "r2_cv": float(r2_score(y_train_fit, cv_predictions)),
        "rmse_test": float(np.sqrt(mean_squared_error(y_test, test_predictions))),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best = results_df.loc[results_df["rmse_cv"].idxmin()]
print(f"Mejor candidato: {best['modelo']} con RMSE CV = {best['rmse_cv']:.4f}")

# ---------- 5. Visualizacion simple (EDA) ----------
plt.figure(figsize=(8, 4))
sns.histplot(df_clean["target"], kde=True)
plt.title("Distribucion del objetivo (progresion de diabetes)")
plt.show()
